# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:  
  Which existing catalog films should a streaming service prioritze licensing to increase subscriber count?  
  
- Why this question matters:  
  It tests if we can make more insightful decisions on which films to license will generate more profit while being more precise on licensing budget.
- Tables/files used:  
  genres, ratings, credits, keywords, links
- Row meaning of each important table:  
  -movies: row representing a single movie  
  -ratings: row represents a single rating by one user for one movie  
  -credits: row represents the cast, crew, and metadata for single movie  
  -keywords: row represents any descreptive plot tags for a single movie  
  -links: row represents a mapping between the movie and MovieID catalogs like IMDb, tmbdID, and MovieLens
- Early data quality issues:  
  -Missing values in keywords, credits, and movies. The dataset also uses two different ID spaces; movieID and tmbID causing identifyer discrepancies.
- Initial cleaning plan:  
  -For my cleaning plan, first I will verify that movieID and tmbdID match across tables and convert dates to datetime format. Second, I will replace zero values in budget and revenue with NaN so missing numbers do not mix in with the financial data. Finally, I will aggregate the ratings and table by movieID, filter for fims with at least 20 ratings, and join the summary back to the movie tables.
- Possible target/outcome:  
  -A numeric variable high_user_rating, evaluated on ly on films with at least 20 ratings. This allows classification model to predict whether an older catelog film has strong enough audience reaction in modern times to justify licensing cost.
- Ethics, privacy, or governance concerns:  
  -User ratings are anonymized, so direct privacy risk is low. However, historical MovieLens ratings reflect the deomgraphic preferences of past users, which can create popularity bias against niche, indie, or international films. Straming licensing decisions should us model predictions as decision support alongside editorial review rather than an automated rule.

In [6]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.

# movies with $0 budget or $0 revenue
print((movies['budget'] == 0).sum())
print((movies['revenue'] == 0).sum())

# TMDbIDs existing in both movies and credits
movies_ids = set(movies['tmdbId'])
credits_ids = set(credits['tmdbId'])

print(len(movies_ids.intersection(credits_ids)))

# ratings per movie and how many have 20 or more
rating_counts = ratings.groupby('movieId')['rating'].count()

print((rating_counts >= 20).sum())



4443
4322
9082
1300


In [7]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


,missing_rate
userId,0.0
movieId,0.0
rating,0.0
timestamp,0.0
rating_datetime,0.0


... (5 outputs left out)

,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis: one row per movie (movieID)
- Primary key for one row: movieID
- Tables aggregated before joining: ratings aggregated to movieID
- Tables joined directly: movies with credits, keywords and the aggregated ratings summary
- Row-count validation after each major merge: Verify after every merge that total rows equal the base movie catalog and movieID uniquness is strictly 1:1
- New variables created: rating_mean, first_rating, last_rating, distinct_users, budget_clean, and revenue_clean.
- Remaining data quality concerns: missing financial metrics, missing keywords for movies and missing TMDB IDs in crosswalk links


In [8]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.
import numpy as np
import pandas as pd

# 1. Aggregate ratings to movie-level granularity (movieId)
ratings_agg = (
    ratings.groupby("movieId")
    .agg(
        rating_mean=("rating", "mean"),
        rating_count=("rating", "count"),
        rating_std=("rating", "std"),
        first_rating=("rating_datetime", "min"),
        last_rating=("rating_datetime", "max"),
        distinct_users=("userId", "nunique"),
    )
    .reset_index()
)

# Fill standard deviation with 0 for movies that only received 1 rating
ratings_agg["rating_std"] = ratings_agg["rating_std"].fillna(0.0)

# 2. Join aggregated ratings into movies
abt = movies.merge(ratings_agg, on="movieId", how="left")

# 3. Join curated movie metadata (credits & keywords on tmdbId)
abt = abt.merge(credits, on="tmdbId", how="left")
abt = abt.merge(keywords, on="tmdbId", how="left")

# 4. Construct target variable (high_user_rating threshold: e.g., mean >= 3.5 for count >= 20)
abt["high_user_rating"] = np.where(
    abt["rating_count"] >= 20, (abt["rating_mean"] >= 3.5).astype(int), np.nan
)

# 5. Clean budget and revenue zeros into NaN
abt["budget_clean"] = abt["budget"].replace(0, np.nan)
abt["revenue_clean"] = abt["revenue"].replace(0, np.nan)

# 6. Validation checks
print(f"Base movies rows: {len(movies)}")
print(f"ABT final rows:   {len(abt)}")
print(f"Is movieId unique in ABT? {abt['movieId'].is_unique}")
assert len(abt) == len(
    movies
), "Row count changed! Check for duplicate keys in joined tables."
assert abt["movieId"].is_unique, "movieId has duplicate rows in ABT!"

abt.head(3)




Base movies rows: 9082
ABT final rows:   9082
Is movieId unique in ABT? True


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,rating_mean,rating_count,rating_std,first_rating,last_rating,distinct_users,director,top_cast,keywords_list,high_user_rating,budget_clean,revenue_clean
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,247.0,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1.0,30000000.0,373554033.0
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,107.0,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,0.0,65000000.0,262797249.0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,59.0,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0.0,NaN,NaN


In [9]:
rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


In [10]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


movie_rows_before_merges             9082
abt_rows_after_merges                9082
duplicate_movie_ids                     0
duplicate_tmdb_ids                      0
movies_with_rating_summary           9025
movies_eligible_for_rating_target    1300
dtype: int64

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,director,top_cast,keywords_list,has_revenue,rating_target_eligible,high_user_rating
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,True,1
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,True,0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,True,0
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,2.384615,13.0,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,False,<NA>
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,3.267857,56.0,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,True,0


,missing_rate
high_user_rating,0.856860
user_rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
last_rating_date,0.006276
user_rating_count,0.006276
user_rating_mean,0.006276
first_rating_date,0.006276
genres_list,0.003854
primary_genre,0.003854


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [11]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

env_path = repo_root / ".env"
env_text = env_path.read_text() if env_path.exists() else ""
if "PROJECT_DATABASE_URL" not in env_text:
    if env_text and not env_text.endswith("\n"):
        env_text += "\n"
    env_text += ("# Your own Neon database (Path A). Paste your connection string after the = sign.\n"
                 "PROJECT_DATABASE_URL=\n")
    env_path.write_text(env_text)
    print("Added an empty PROJECT_DATABASE_URL line to .env")

load_dotenv(env_path, override=True)
project_url = os.getenv("PROJECT_DATABASE_URL")

SQL_RESULTS_DIR = repo_root / "outputs" / "sql_results"
SQL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if project_url:
    project_engine = create_engine(project_url)
    with project_engine.connect() as conn:
        conn.execute(text("SELECT 1;"))
    print("Connected to your own database. For safety, we do not print the connection string.")
else:
    project_engine = None
    print("PROJECT_DATABASE_URL is empty, so the database steps will be skipped.")
    print("Path A: open .env, paste your own Neon connection string after PROJECT_DATABASE_URL=, save, and run this cell again.")

Connected to your own database. For safety, we do not print the connection string.


In [12]:
def upload_table(source, table_name):
    if project_engine is None:
        print(f"Skipped uploading {table_name}: PROJECT_DATABASE_URL is not set.")
        return
    df = pd.read_csv(source) if isinstance(source, (str, Path)) else source.copy()
    df.to_sql(table_name, project_engine, if_exists="replace", index=False)
    with project_engine.connect() as conn:
        db_rows = conn.execute(text(f'SELECT COUNT(*) FROM "{table_name}";')).scalar()
    status = "OK" if db_rows == len(df) else "MISMATCH"
    print(f"Uploaded {table_name}: {len(df)} rows in Python, {db_rows} rows in database ({status})")

upload_table(abt, "movies_abt")

Uploaded movies_abt: 9082 rows in Python, 9082 rows in database (OK)


In [13]:
def run_project_sql(query, result_name):
    csv_path = SQL_RESULTS_DIR / f"{result_name}.csv"
    if project_engine is not None:
        with project_engine.connect() as conn:
            result = pd.read_sql_query(text(query), conn)
        result.to_csv(csv_path, index=False)
        print(f"Ran on your database. Saved the result to {csv_path.relative_to(repo_root)}")
        return result
    if csv_path.exists():
        print(f"Using saved result from {csv_path.relative_to(repo_root)} (no database connection needed).")
        return pd.read_csv(csv_path)
    print(f"Skipped {result_name}: no database connection and no saved CSV yet.")
    return None

abt_validation_query = """
SELECT
    COUNT(*) AS abt_rows,
    COUNT(DISTINCT "movieId") AS unique_movie_ids,
    COUNT(user_rating_count) AS movies_with_any_ratings,
    COUNT(high_user_rating) AS movies_with_20plus_ratings,
    SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
FROM movies_abt;
"""

run_project_sql(abt_validation_query, "abt_key_and_target_check")

Ran on your database. Saved the result to outputs/sql_results/abt_key_and_target_check.csv


,abt_rows,unique_movie_ids,movies_with_any_ratings,movies_with_20plus_ratings,target_not_defined_rows
0,9082,9082,9025,1300,7782


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [14]:
data_dict_records = [
    {
        "column_name": "movieId",
        "data_type": "int64",
        "description": "Unique identifier for each movie in MovieLens",
        "source": "movies",
        "transformation_or_derivation": "Primary key from base movies catalog",
        "notes_or_concerns": "Strictly unique 1:1 row identifier across all 9,082 titles",
    },
    {
        "column_name": "tmdbId",
        "data_type": "int64",
        "description": "The Movie Database (TMDb) identifier",
        "source": "movies / links",
        "transformation_or_derivation": "Crosswalk foreign key used to join credits and keywords",
        "notes_or_concerns": "Used for table joins; 0.14% missing in original raw links table",
    },
    {
        "column_name": "title",
        "data_type": "str",
        "description": "Official theatrical release title",
        "source": "movies",
        "transformation_or_derivation": "Direct extract from base movies",
        "notes_or_concerns": "May include non-ASCII characters; used for catalog reference",
    },
    {
        "column_name": "release_year",
        "data_type": "int64",
        "description": "Four-digit release year of the film",
        "source": "movies",
        "transformation_or_derivation": "Extracted from release_date datetime",
        "notes_or_concerns": "Used for historical catalog cohort segmentation",
    },
    {
        "column_name": "primary_genre",
        "data_type": "str",
        "description": "Primary genre classification",
        "source": "movies",
        "transformation_or_derivation": "Extracted leading genre tag from JSON array",
        "notes_or_concerns": "Sparse missingness (0.38%); 19 distinct categorical genres",
    },
    {
        "column_name": "user_rating_mean",
        "data_type": "float64",
        "description": "Average audience rating on a 0.5 to 5.0 scale",
        "source": "ratings",
        "transformation_or_derivation": "ratings.groupby('movieId')['rating'].mean()",
        "notes_or_concerns": "Unstable for films with very few ratings",
    },
    {
        "column_name": "user_rating_count",
        "data_type": "float64",
        "description": "Total count of user ratings received",
        "source": "ratings",
        "transformation_or_derivation": "ratings.groupby('movieId')['rating'].count()",
        "notes_or_concerns": "Evaluation filter metric; requires >= 20 ratings for target eligibility",
    },
    {
        "column_name": "user_rating_std",
        "data_type": "float64",
        "description": "Standard deviation of user ratings",
        "source": "ratings",
        "transformation_or_derivation": "ratings.groupby('movieId')['rating'].std()",
        "notes_or_concerns": "Missing (NaN) for single-rating films (34.1% of raw catalog)",
    },
    {
        "column_name": "director",
        "data_type": "str",
        "description": "Primary director name",
        "source": "credits",
        "transformation_or_derivation": "Extracted from credits crew array where job == 'Director'",
        "notes_or_concerns": "Joined on tmdbId; 0.26% missing",
    },
    {
        "column_name": "top_cast",
        "data_type": "str",
        "description": "Pipe-delimited top 3 billed actors",
        "source": "credits",
        "transformation_or_derivation": "Extracted first 3 names from credits cast array",
        "notes_or_concerns": "Joined on tmdbId; 0.99% missing",
    },
    {
        "column_name": "keywords_list",
        "data_type": "str",
        "description": "Pipe-delimited plot and descriptive keywords",
        "source": "keywords",
        "transformation_or_derivation": "Concatenated keyword names from TMDb keywords table",
        "notes_or_concerns": "Missing for ~8.38% of titles",
    },
    {
        "column_name": "high_user_rating",
        "data_type": "Int64",
        "description": "Binary target indicator (1 = high rating, 0 = low rating)",
        "source": "ratings derivation",
        "transformation_or_derivation": "Derived as 1 if user_rating_mean >= 3.5 for count >= 20; NaN otherwise",
        "notes_or_concerns": "Target outcome for classification modeling; defined for 1,300 eligible films",
    },
]

data_dictionary = pd.DataFrame(data_dict_records)
data_dictionary


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,Unique identifier for each movie in MovieLens,movies,Primary key from base movies catalog,Strictly unique 1:1 row identifier across all ...
1,tmdbId,int64,The Movie Database (TMDb) identifier,movies / links,Crosswalk foreign key used to join credits and...,Used for table joins; 0.14% missing in origina...
2,title,str,Official theatrical release title,movies,Direct extract from base movies,May include non-ASCII characters; used for cat...
3,release_year,int64,Four-digit release year of the film,movies,Extracted from release_date datetime,Used for historical catalog cohort segmentation
4,primary_genre,str,Primary genre classification,movies,Extracted leading genre tag from JSON array,Sparse missingness (0.38%); 19 distinct catego...
5,user_rating_mean,float64,Average audience rating on a 0.5 to 5.0 scale,ratings,ratings.groupby('movieId')['rating'].mean(),Unstable for films with very few ratings
6,user_rating_count,float64,Total count of user ratings received,ratings,ratings.groupby('movieId')['rating'].count(),Evaluation filter metric; requires >= 20 ratin...
7,user_rating_std,float64,Standard deviation of user ratings,ratings,ratings.groupby('movieId')['rating'].std(),Missing (NaN) for single-rating films (34.1% o...
8,director,str,Primary director name,credits,Extracted from credits crew array where job ==...,Joined on tmdbId; 0.26% missing
9,top_cast,str,Pipe-delimited top 3 billed actors,credits,Extracted first 3 names from credits cast array,Joined on tmdbId; 0.99% missing


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [15]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [16]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,6002,1994-01-01,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298936


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [17]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [18]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
